In [1]:
import numpy as np
import torch
import random

# =====================================================================
# STEP 1: INITIAL DATA LOADING & HARDWARE CHECK
# =====================================================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"CUDA status: {torch.cuda.is_available()}")
print(f"Running on GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

# Load the archive (ensure factorizations_r.npz is in your directory!)
data = np.load('factorizations_r.npz', allow_pickle=True)
U_23, V_23, W_23 = data['3,3,3']

# Convert numpy base matrices to PyTorch tensors
U_base = torch.tensor(U_23, dtype=torch.float32)
V_base = torch.tensor(V_23, dtype=torch.float32)
W_base = torch.tensor(W_23, dtype=torch.float32)

# =====================================================================
# STEP 2: DEFINE THE PERFECT 3X3 TARGET TENSOR
# =====================================================================
T_target = torch.zeros((9, 9, 9), dtype=torch.float32).to(device)
for i in range(3):
    for j in range(3):
        for k in range(3):
            a_idx = i * 3 + j
            b_idx = j * 3 + k
            c_idx = i * 3 + k
            T_target[a_idx, b_idx, c_idx] = 1.0

# =====================================================================
# STEP 3: AUTOMATED MULTI-START RANDOMIZED SEARCH LOOP
# =====================================================================
max_search_attempts = 5000  # Leave running to test thousands of variations
epochs_per_run = 5000       # We prune early, so we don't need 50k epochs if it's stuck

print("\n🚀 Starting Multi-Start Random Search... 🚀")
print("This script will run indefinitely, testing random variations until it hits zero loss.")
print("Press 'Interrupt' in Jupyter at any time to stop it.\n")

for attempt in range(1, max_search_attempts + 1):
    # Randomly select which of the 23 columns to drop for this run
    drop_idx = random.randint(0, 22)
    indices_to_keep = [idx for idx in range(23) if idx != drop_idx]
    
    # Create the 22-step matrix copies
    U_22 = U_base[:, indices_to_keep].clone().to(device)
    V_22 = V_base[:, indices_to_keep].clone().to(device)
    W_22 = W_base[:, indices_to_keep].clone().to(device)
    
    # INJECT RANDOM NOISE:
    # We add a tiny bit of random variation to kick it out of the original crater
    noise_scale = 0.15
    U_22 = (U_22 + torch.randn_like(U_22) * noise_scale).requires_grad_(True)
    V_22 = (V_22 + torch.randn_like(V_22) * noise_scale).requires_grad_(True)
    W_22 = (W_22 + torch.randn_like(W_22) * noise_scale).requires_grad_(True)
    
    optimizer = torch.optim.Adam([U_22, V_22, W_22], lr=0.01)
    
    best_loss = 1.0
    
    # Fast training loop
    for epoch in range(epochs_per_run):
        optimizer.zero_grad()
        
        # Fast tensor reconstruction on GPU
        T_pred = torch.einsum('ir,jr,kr->ijk', U_22, V_22, W_22)
        
        # Calculate loss
        loss = torch.nn.functional.mse_loss(T_pred, T_target)
        
        loss.backward()
        optimizer.step()
        
        if loss.item() < best_loss:
            best_loss = loss.item()
            
        # --- EARLY STOPPING (PRUNING) ---
        # If we are at epoch 1500 and the loss hasn't dropped below 0.002,
        # it means we are in a bad crater. Kill this run and try a new random layout!
        if epoch == 1500 and best_loss > 0.002:
            break
            
        # THE WORLD RECORD WIN CONDITION
        if loss.item() < 1e-6:
            print(f"\n🎉 SUCCESS! 🎉")
            print(f"A 22-step solution was discovered on Attempt #{attempt}!")
            print(f"Dropped column index: {drop_idx}")
            
            # Save our world record to disk!
            np.savez('WORLD_RECORD_22_STEP.npz', 
                     u=U_22.detach().cpu().numpy(), 
                     v=V_22.detach().cpu().numpy(), 
                     w=W_22.detach().cpu().numpy())
            break
            
    # Print status updates every 100 attempts
    if attempt % 50 == 0 or best_loss < 0.001:
        print(f"Attempt {attempt:04d}/{max_search_attempts} | Dropped: {drop_idx:02d} | Best MSE: {best_loss:.6f}")
        
    if best_loss < 1e-6:
        break

print("\nSearch Loop Terminated.")

CUDA status: True
Running on GPU: NVIDIA GeForce GTX 1080 Ti

🚀 Starting Multi-Start Random Search... 🚀
This script will run indefinitely, testing random variations until it hits zero loss.
Press 'Interrupt' in Jupyter at any time to stop it.

Attempt 0001/5000 | Dropped: 22 | Best MSE: 0.000062
Attempt 0003/5000 | Dropped: 11 | Best MSE: 0.000093
Attempt 0006/5000 | Dropped: 09 | Best MSE: 0.000028
Attempt 0008/5000 | Dropped: 19 | Best MSE: 0.000069
Attempt 0014/5000 | Dropped: 21 | Best MSE: 0.000103
Attempt 0016/5000 | Dropped: 15 | Best MSE: 0.000064
Attempt 0019/5000 | Dropped: 03 | Best MSE: 0.000055
Attempt 0020/5000 | Dropped: 14 | Best MSE: 0.000053
Attempt 0022/5000 | Dropped: 13 | Best MSE: 0.000318
Attempt 0026/5000 | Dropped: 17 | Best MSE: 0.000042
Attempt 0029/5000 | Dropped: 01 | Best MSE: 0.000136
Attempt 0032/5000 | Dropped: 17 | Best MSE: 0.000130
Attempt 0033/5000 | Dropped: 01 | Best MSE: 0.000145
Attempt 0036/5000 | Dropped: 02 | Best MSE: 0.000067
Attempt 0037/5

KeyboardInterrupt: 